#Install Unsloth, Pypdf & Dataset

* Make fine tuning 2x Faster.
* 50% less memory.

In [1]:
!pip install unsloth pypdf datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 869.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2

# import liabarie

In [10]:
from pypdf import PdfReader
import io
import requests
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# PDF Text Extraction

In [2]:
url = "https://home-wordpress.deeplearning.ai/wp-content/uploads/2022/03/andrew-ng-machine-learning-yearning.pdf"

In [3]:
response = requests.get(url)
reader = PdfReader(io.BytesIO(response.content))

text = ""
for page in reader.pages:
    text += page.extract_text()

print(f"Total Pages: {len(reader.pages)}")
print("\n Starting 500 characters:")
print(text[:500])

Total Pages: 118

 Starting 500 characters:
 
 
 
 
 
 
 
Machine Learning Yearning is a  
deeplearning.ai project. 
 
 
 
 
 
 
 
 
 
 
© 2018 Andrew Ng. All Rights Reserved. 
 
  
Page 2 Machine Learning Yearning-Draft Andrew Ng 
Deeplearning.AI 
Table of Contents 
 
1 Why Machine Learning Strategy 
2 How to use this book to help your team 
3 Prerequisites and Notation 
4 Scale drives machine learning progress 
5 Your development and test sets 
6 Your dev and test sets should come from the same distribution 
7 How large do the dev/test 


# Chunking

In [4]:
# split combined data to chunks
def split_into_chunks(text, chunk_size=500):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

In [5]:
chunks = split_into_chunks(text)
print(f"Total Chunks: {len(chunks)} ")
print("\n First Chunk:")
print(chunks[0])

Total Chunks: 52 

 First Chunk:
Machine Learning Yearning is a deeplearning.ai project. © 2018 Andrew Ng. All Rights Reserved. Page 2 Machine Learning Yearning-Draft Andrew Ng Deeplearning.AI Table of Contents 1 Why Machine Learning Strategy 2 How to use this book to help your team 3 Prerequisites and Notation 4 Scale drives machine learning progress 5 Your development and test sets 6 Your dev and test sets should come from the same distribution 7 How large do the dev/test sets need to be? 8 Establish a single-number evaluation metric for your team to optimize 9 Optimizing and satisficing metrics 10 Having a dev set and metric speeds up iterations 11 When to change dev/test sets and metrics 12 Takeaways: Setting up development and test sets 13 Build your first system quickly, then iterate 14 Error analysis: Look at dev set examples to evaluate ideas 15 Evaluating multiple ideas in parallel during error analysis 16 Cleaning up mislabeled dev and test set examples 17 If you have a large

# Making dataset From Chunked Data

In [6]:
from datasets import Dataset

In [7]:
dataset_list = []

for chunk in chunks:
    dataset_list.append({
        "text": f"""### Instruction:
Neeche diye gaye text ko samjho aur yaad rakho:

### Input:
{chunk}

### Response:
Main ne yeh information seekh li hai."""
    })

hf_dataset = Dataset.from_list(dataset_list)
print(f"Dataset Ready! ")
print(f"Total Examples: {len(hf_dataset)}")
print("\nFirst Example:")
print(hf_dataset[0]['text'][:300])

Dataset Ready! 
Total Examples: 52

First Example:
### Instruction:
Neeche diye gaye text ko samjho aur yaad rakho:

### Input:
Machine Learning Yearning is a deeplearning.ai project. © 2018 Andrew Ng. All Rights Reserved. Page 2 Machine Learning Yearning-Draft Andrew Ng Deeplearning.AI Table of Contents 1 Why Machine Learning Strategy 2 How to use 


## Loading Model From Hugging Face Using Unsloth

In [8]:
# load model and tokenizer (LORA)
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "unsloth/Llama-3.2-1B",
#     max_seq_length = 2048,   #how muchwords process at a time
#     load_in_4bit = False,  # LoRA mode
# )

# print("LORA Model Loaded Successfully")

# load model and tokenizer (QLORA)
qlora_model, qlora_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B",
    max_seq_length = 2048,
    load_in_4bit = True,  # QLORA in 4 bit
)

print("QLORA Model Loaded Successfully")

==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.


QLORA Model Loaded Successfully


## Apply LORA & QLORA

In [9]:
# LORA MODEL
# model = FastLanguageModel.get_peft_model(
#     model,
#     r = 16, # rank
#     target_modules = ["q_proj", "v_proj"], # 2 layers finetune from multiple layers
#     lora_alpha = 16, # learning seed
#     lora_dropout = 0, #stop overfitting
#     bias = "none", # baise weights
# )

# print("LORA Model Applied Successfully")

# QLORA MODEL
qlora_model = FastLanguageModel.get_peft_model(
    qlora_model,
    r = 16,
    target_modules = ["q_proj", "v_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Extra memory saving for QLORA
)

print("QLoRA Model Applied Successfully! ")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch Attention layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.5.10 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


QLoRA Model Applied Successfully! 


# Model Training (LORA & QLORA)

In [11]:
# LORA
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = hf_dataset,
#     dataset_text_field = "text",
#     max_seq_length = 2048,
#     args = TrainingArguments(
#         per_device_train_batch_size = 2,
#         gradient_accumulation_steps = 4,
#         warmup_steps = 10,
#         max_steps = 100,
#         learning_rate = 2e-4,
#         output_dir = "lora_model",
#     ),
# )

# print("LORA Trainer Ready! ")

# QLORA


qlora_trainer = SFTTrainer(
    model = qlora_model,           # qlora_model
    tokenizer = qlora_tokenizer,   # qlora_tokenizer
    train_dataset = hf_dataset,    # same dataset
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 2e-4,
        output_dir = "qlora_model",  # alag folder
    ),
)

print("QLoRA Trainer Ready! ")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/52 [00:00<?, ? examples/s]

QLoRA Trainer Ready! 


In [12]:
# start train LORA
#trainer.train()

# start train LORA
qlora_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 52 | Num Epochs = 15 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,703,936 of 1,237,518,336 (0.14% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in qlora_model/checkpoint-100/tokenizer_config.json.


TrainOutput(global_step=100, training_loss=2.161001739501953, metrics={'train_runtime': 199.7559, 'train_samples_per_second': 4.005, 'train_steps_per_second': 0.501, 'total_flos': 2984830971568128.0, 'train_loss': 2.161001739501953, 'epoch': 14.307692307692308})

# Testing LORA

In [14]:
# FastLanguageModel.for_inference(model)

# inputs = tokenizer(
# """### Instruction:
# Machine learning kya hai?

# ### Response:
# """, return_tensors = "pt").to("cuda")

In [15]:

# outputs = model.generate(**inputs, max_new_tokens = 100)
# print(tokenizer.decode(outputs[0], skip_special_tokens = True))

# Testing QLORA

In [16]:
# QLoRA Test
FastLanguageModel.for_inference(qlora_model)
inputs = qlora_tokenizer("""### Instruction:
What is machine learning?

### Response:
""", return_tensors="pt").to("cuda")
qlora_output = qlora_model.generate(**inputs, max_new_tokens=100)
qlora_answer = qlora_tokenizer.decode(qlora_output[0], skip_special_tokens=True)


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

In [17]:
print(qlora_answer)

### Instruction:
What is machine learning?

### Response:
Machine learning is the subset of AI that uses algorithms to build systems that learn from data. It is a subset of AI because it uses algorithms. Machine learning can be contrasted with other AI subfields such as natural language processing (NLP), computer vision, and speech recognition. These other AI subfields use algorithms to build systems that are very good at performing certain tasks, but not as good at learning from data. For example, NLP systems are very good at understanding what is in a picture
